In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

for lib in ["nltk", "wordcloud", "textblob", "sklearn"]:
    try:
        __import__(lib)
        print(lib, "OK")
    except ImportError:
        print(lib, "MISSING")

In [ ]:
import nltk
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("wordnet")

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

df = pd.read_csv("twitter_training.csv", header=None,
                 names=["tweet_id", "entity", "sentiment", "text"])
df.head()

In [ ]:
print("Shape:", df.shape)
print()
df.info()
print()
print("Nulls per column:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate texts:", df["text"].duplicated().sum())
print()
print(df["sentiment"].value_counts())

In [ ]:
df["sentiment"] = df["sentiment"].replace("Irrelevant", "Neutral")

print("Start:", len(df))
df = df.dropna(subset=["text"])
print("After dropping missing text:", len(df))
df = df.drop_duplicates(subset="text")
print("After dropping duplicate texts:", len(df))

print(df["sentiment"].value_counts())
print((df["sentiment"].value_counts(normalize=True) * 100).round(1))

In [ ]:
order = ["Negative", "Neutral", "Positive"]
palette = {"Negative": "#d9534f", "Neutral": "#999999", "Positive": "#5cb85c"}

plt.figure(figsize=(6,4))
sns.countplot(x="sentiment", data=df, order=order, hue="sentiment", palette=palette, legend=False)
plt.title("Sentiment Distribution")
plt.show()

print(df["sentiment"].value_counts())

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

stop = set(stopwords.words("english")) - {"not", "no", "nor"}   # keep negations
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # URLs
    text = re.sub(r"@\w+", " ", text)               # mentions
    text = re.sub(r"[^a-z\s]", " ", text)           # punctuation, digits, emojis
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop and len(w) > 1]
    return " ".join(tokens)

df["clean_text"] = df["text"].apply(preprocess)
df = df[df["clean_text"].str.strip() != ""].reset_index(drop=True)

print("Rows after preprocessing:", len(df))
df[["text", "clean_text"]].sample(5, random_state=1)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, df["sentiment"], groups=df["tweet_id"]))
train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]

print("Train:", len(train_df), "| Test:", len(test_df), "| Test share:", round(len(test_df)/len(df), 3))
print("Shared tweet_ids between train and test:", len(set(train_df["tweet_id"]) & set(test_df["tweet_id"])))
print(train_df["sentiment"].value_counts(normalize=True).round(3))
print(test_df["sentiment"].value_counts(normalize=True).round(3))

## Train/Test Split

I split the data 80/20 by tweet_id (55,209 training and 13,852 test tweets). Each tweet appears several times with slightly changed wording, and a random split would put one version in training and another in test, which would make the scores look better than they are. No tweet_id appears in both sets, and the class percentages are similar in both (Neutral about 41-42%, Negative 30-32%, Positive 27%).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df=3)
X_train = tfidf.fit_transform(train_df["clean_text"])
X_test = tfidf.transform(test_df["clean_text"])
y_train, y_test = train_df["sentiment"], test_df["sentiment"]

print("Train matrix:", X_train.shape, "| Test matrix:", X_test.shape)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nb = MultinomialNB()
lr = LogisticRegression(max_iter=1000)
nb.fit(X_train, y_train)
lr.fit(X_train, y_train)

preds = {"Naive Bayes": nb.predict(X_test), "Logistic Regression": lr.predict(X_test)}
labels = ["Negative", "Neutral", "Positive"]

for name, p in preds.items():
    print("=" * 55)
    print(name, "| accuracy:", round(accuracy_score(y_test, p), 3))
    print(classification_report(y_test, p, labels=labels))

## Model Evaluation

| Model | Accuracy | Macro F1 |
|---|---|---|
| Naive Bayes | 0.605 | 0.60 |
| Logistic Regression | 0.609 | 0.60 |

The two models perform almost the same. Logistic Regression finds more positive tweets (recall 0.53 vs 0.46), while Naive Bayes finds more neutral ones (recall 0.70 vs 0.65). In both confusion matrices the most common mistakes are Positive or Negative tweets predicted as Neutral, and confusion between Positive and Negative is much rarer. So the models usually detect the direction of the sentiment but find it hard to separate it from neutral text.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11,4))
for a, (name, p) in zip(ax, preds.items()):
    cm = confusion_matrix(y_test, p, labels=labels)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=a, cbar=False)
    a.set_title(name)
    a.set_xlabel("Predicted")
    a.set_ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
lr_pred = preds["Logistic Regression"]
errors = test_df.assign(predicted=lr_pred)
errors = errors[errors["sentiment"] != errors["predicted"]]
print("Misclassified:", len(errors), "of", len(test_df))

pd.set_option("display.max_colwidth", 200)
errors[["text", "sentiment", "predicted"]].sample(5, random_state=7).rename(columns={"sentiment": "actual"})

## Error Analysis

Logistic Regression misclassified 5,411 of 13,852 test tweets (39.1%). In the five examples I checked, four were Positive tweets predicted as Neutral. The reasons I can see are:
- Weak or indirect sentiment, where the tweet expresses opinion without clear sentiment words (for example "Threaded carbine is cute" and a question about favourite gifts).
- Sarcasm or long phrases the bag-of-words model cannot understand (the COVID vaccine tweet).
- Possible label noise, such as a tweet full of positive words ("fun", "thanks") that is labelled Neutral.
- Lists of names (the World of Warcraft raids tweet), where the neutral-looking words hide the one positive word.
TF-IDF treats words independently, so it cannot capture tone or context.

In [ ]:
from wordcloud import WordCloud

colors = {"Negative": "Reds", "Neutral": "Greys", "Positive": "Greens"}
fig, ax = plt.subplots(1, 3, figsize=(18,5))
for a, s in zip(ax, ["Negative", "Neutral", "Positive"]):
    text = " ".join(train_df.loc[train_df["sentiment"] == s, "clean_text"])
    wc = WordCloud(width=600, height=400, background_color="white", colormap=colors[s], max_words=80).generate(text)
    a.imshow(wc, interpolation="bilinear")
    a.set_title(s)
    a.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
names = np.array(tfidf.get_feature_names_out())
for i, cls in enumerate(lr.classes_):
    top = names[np.argsort(lr.coef_[i])[-10:][::-1]]
    print(cls, ":", ", ".join(top))

## Word Clouds and Top Words

The most influential words for Negative tweets are fix, suck, glitch, server, bug and worst, and for Positive tweets fun, love, excited, thank, best and awesome, so the model has learned sensible sentiment words. The Neutral class is identified by words like com, dlvr, group and viewer, which come from links and promotional posts, so the model partly recognises Neutral by tweet style and not by meaning. The word clouds contain words such as "unk", "pic" and "twitter" in all three classes. These come from placeholder tokens and picture links, and they carry no sentiment.

## Conclusion

Logistic Regression (accuracy 0.609, macro F1 0.60) and Naive Bayes (0.605, 0.60) performed almost the same, so the difference is too small to be a clear winner. I would choose Logistic Regression because it is slightly more balanced and finds more positive tweets (recall 0.53 vs 0.46), and it gives interpretable word weights. Both models are far above random guessing (33%) and always predicting Neutral (41%), but a 61% accuracy is only moderate.

Possible real-world use: monitoring customer or player feedback about games and brands, for example flagging negative tweets about bugs and server problems for a support team.

Limitations: the three classes include merged "Irrelevant" tweets in Neutral, some labels look noisy, the models cannot understand sarcasm or context, and leftover tokens such as "unk" and "pic" add noise. Improvements could include word embeddings or a transformer model, and removing the leftover tokens.